# Japan Cherry — Yedoensis vs Sargentii (draft analysis)

Runs a small BloomBench-style sweep on two GMU-Japan subsets:

* `GMU_Cherry_Japan_Y` — Cerasus yedoensis (Somei-yoshino) locations only
* `GMU_Cherry_Japan_S` — Cerasus sargentii locations only

Each dataset is fitted with the following baselines and the test-set MAE is reported:

| Key | Class |
|---|---|
| Mean | `MeanModel` |
| Linear | `LinearTrendModel` (per-species linear trend of DOY vs year) |
| RandomForest | `RandomForestModel` |
| XGBoost | `XGBoostModel` |
| CNN1D | `CNN1DModel` |
| LSTM | `LSTMModel` |
| Transformer | `TransformerModel` |

**Split.** Temporal cutoff at the 75th-percentile year (`--split_years cutoff --split_years_size 0.75`).

**Features.** Daily mean 2 m temperature only (OpenMeteo / ERA5).

**Target.** GMU peak-bloom day code `gmu_0` (Japan).

## 1. Config

In [ ]:
from __future__ import annotations

import os
import time
import warnings
from typing import Any, Callable, Dict, Tuple

# Targeted suppression of one specific sklearn 1.8 warning. It is emitted from
# inside sklearn's own parallel machinery (sklearn/utils/parallel.py:144) when
# `sklearn.utils.parallel.delayed` is paired with plain `joblib.Parallel`
# instead of sklearn's matching `Parallel` wrapper. The warning is about
# propagating `sklearn.set_config(...)` and warning filters into workers —
# pysephone never calls `sklearn.set_config`, so this is noise, not a
# correctness issue. We silence ONLY this message; all other UserWarnings
# remain visible.
#
# The `message` argument to `warnings.filterwarnings` is a regex anchored at
# the start of the message (via `re.match`). The actual sklearn message begins
# with a backtick (`` `sklearn.utils.parallel.delayed` should ... ``), so the
# regex must include it. PYTHONWARNINGS is also set so the filter propagates
# into joblib worker subprocesses.
_SKLEARN_DELAYED_RE = r'`sklearn\.utils\.parallel\.delayed`'
warnings.filterwarnings('ignore', message=_SKLEARN_DELAYED_RE, category=UserWarning)
os.environ['PYTHONWARNINGS'] = f'ignore:{_SKLEARN_DELAYED_RE}:UserWarning'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pysephone.constants import KEY_OBSERVATIONS
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.evaluation.regression import SingleTargetRegression

from pysephone.models.mean import MeanModel
from pysephone.models.linear_trend import LinearTrendModel
from pysephone.models.random_forest import RandomForestModel
from pysephone.models.xgb import XGBoostModel
from pysephone.models.cnn_1d import CNN1DModel
from pysephone.models.lstm import LSTMModel
from pysephone.models.transformer import TransformerModel

In [ ]:
# Datasets to compare (target obs-type code, registry key)
DATASETS = [
    ('GMU_Cherry_Japan_Y', 'gmu_0'),  # yedoensis
    ('GMU_Cherry_Japan_S', 'gmu_0'),  # sargentii
]

# Features and season window
FEATURE_KEYS  = ['temperature_2m_mean']
SEASON_START  = '10-01'
SEASON_LENGTH = 365

# Temporal split: 75% of years for train, the rest for test
SPLIT_SIZE = 0.75

# Reproducibility
SEED = 0

# Device for torch models (CNN1D / LSTM / Transformer). Falls back to CPU if
# CUDA is not available — keeps the notebook portable.
import torch as _torch
TORCH_DEVICE = 'cuda' if _torch.cuda.is_available() else 'cpu'

# Model caching: when False, reuse previously-saved fitted models. Set to True
# to ignore the cache and refit everything from scratch. Use
# ``FORCE_RETRAIN_MODELS`` to refit only specific models (matches MODELS keys,
# e.g. {'LSTM', 'Transformer'}); takes precedence over FORCE_RETRAIN_ALL=False.
FORCE_RETRAIN_ALL: bool = False
FORCE_RETRAIN_MODELS: set[str] = set()

# Hyperparameter tuning.
#   RUN_HPO_TREES / RUN_HPO_TORCH: master switches per family.
#   HPO_N_ITER_TREES:    n_iter for RandomizedSearchCV (RF/XGBoost).
#   HPO_N_TRIALS_TORCH:  n random-search trials per torch model.
#   HPO_CV_FOLDS:        GroupKFold(year) folds inside the tree search.
#   HPO_VAL_FRACTION:    last N% of training years used as the cutoff val set
#                        scored by MAE during torch random search.
#   FORCE_RETUNE_*:      ignore the hyperparameter cache and re-run HPO.
# Hyperparameter cache lives at
# ``<data_root>/outputs/hyperparams/<dataset>_<model>.json`` and is keyed by
# ``(dataset, model)`` — NOT by seed. The full per-trial log for torch models
# lives at ``<dataset>_<model>_trials.json`` next to it.
RUN_HPO_TREES: bool = True
RUN_HPO_TORCH: bool = True
HPO_N_ITER_TREES: int = 20
HPO_N_TRIALS_TORCH: int = 15
HPO_CV_FOLDS: int = 5
HPO_VAL_FRACTION: float = 0.2
FORCE_RETUNE_ALL: bool = False
FORCE_RETUNE_MODELS: set[str] = set()

# Hyperparameters for the deep models. Modest defaults so the notebook runs end-to-end in a
# reasonable time.  The manuscript uses 1000 epochs with patience-5 / min-delta 1e-3 early
# stopping; the values below are scaled down for the draft.
TORCH_TRAIN_KWARGS = dict(
    num_epochs=200,
    batch_size=32,
    val_period=10,
    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_min_delta=1e-3,
    device=TORCH_DEVICE,
)

print('Datasets to run:      ', [name for name, _ in DATASETS])
print('Features:             ', FEATURE_KEYS)
print('Train fraction:       ', SPLIT_SIZE)
print('Seed:                 ', SEED)
print('Torch device:         ', TORCH_DEVICE)
print('Force retrain all:    ', FORCE_RETRAIN_ALL)
print('Force retrain subset: ', sorted(FORCE_RETRAIN_MODELS) or '(none)')
print('Run HPO trees/torch:  ', RUN_HPO_TREES, '/', RUN_HPO_TORCH)
print('HPO budgets:           trees n_iter=', HPO_N_ITER_TREES,
      ' folds=', HPO_CV_FOLDS,
      ' / torch trials=', HPO_N_TRIALS_TORCH, sep='')
print('Torch val fraction:   ', HPO_VAL_FRACTION)
print('Force retune all:     ', FORCE_RETUNE_ALL)
print('Force retune subset:  ', sorted(FORCE_RETUNE_MODELS) or '(none)')

## 2. Model registry

Each entry is `(label, fit_callable)`.  Every fit callable takes `(target_fn, dataset_train, seed)` and returns a fitted model instance.  Keeping this as plain callables avoids having to teach the notebook the calling convention of every baseline.

In [ ]:
import json
from pathlib import Path
from sklearn.model_selection import GroupKFold

from pysephone.constants import KEY_YEAR
from pysephone.paths import get_data_root

# ---------------------------------------------------------------------------
# Hyperparameter cache (best_params + per-trial log)
# ---------------------------------------------------------------------------

HP_CACHE_DIR: Path = get_data_root() / 'outputs' / 'hyperparams'
HP_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _hp_cache_path(dataset_name: str, model_key: str) -> Path:
    return HP_CACHE_DIR / f'{dataset_name}_{model_key}.json'


def _hp_trials_path(dataset_name: str, model_key: str) -> Path:
    return HP_CACHE_DIR / f'{dataset_name}_{model_key}_trials.json'


def _json_safe(obj):
    # Make numpy / torch scalars json-encodable.
    if hasattr(obj, 'item'):
        return obj.item()
    return obj


def load_hp_cache(dataset_name: str, model_key: str) -> Dict[str, Any] | None:
    p = _hp_cache_path(dataset_name, model_key)
    if not p.exists():
        return None
    try:
        with open(p) as f:
            return json.load(f)
    except Exception as exc:
        print(f'    HP cache read failed for {p.name}: {type(exc).__name__}: {exc}')
        return None


def save_hp_cache(dataset_name: str, model_key: str, params: Dict[str, Any]) -> None:
    p = _hp_cache_path(dataset_name, model_key)
    with open(p, 'w') as f:
        json.dump(params, f, indent=2, default=_json_safe)


def save_hp_trials(dataset_name: str, model_key: str, trials: list) -> None:
    p = _hp_trials_path(dataset_name, model_key)
    with open(p, 'w') as f:
        json.dump(trials, f, indent=2, default=_json_safe)


def load_hp_trials(dataset_name: str, model_key: str) -> list | None:
    p = _hp_trials_path(dataset_name, model_key)
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)


def _should_retune(model_key: str) -> bool:
    return FORCE_RETUNE_ALL or model_key in FORCE_RETUNE_MODELS


# ---------------------------------------------------------------------------
# Tree HPO (RandomForest / XGBoost): RandomizedSearchCV with year-aware
# GroupKFold. Implementation lives in the model.fit() method itself; here we
# just dispatch around the HP cache.
# ---------------------------------------------------------------------------

def _maybe_run_tree_hpo(model_cls, target_fn, ds_train, seed, dataset_name, model_key):
    cached = None if _should_retune(model_key) else load_hp_cache(dataset_name, model_key)
    if cached is not None:
        model_kwargs = dict(data_keys=FEATURE_KEYS, random_state=seed, **cached)
        model, _ = model_cls.fit(
            target_fn=target_fn, dataset=ds_train,
            model_kwargs=model_kwargs,
            hyperparameter_search=False, random_state=seed,
        )
        return model

    if not RUN_HPO_TREES:
        model_kwargs = dict(data_keys=FEATURE_KEYS, random_state=seed)
        model, _ = model_cls.fit(
            target_fn=target_fn, dataset=ds_train,
            model_kwargs=model_kwargs,
            hyperparameter_search=False, random_state=seed,
        )
        return model

    # Fresh search with year-aware CV
    model_kwargs = dict(data_keys=FEATURE_KEYS, random_state=seed)
    model, info = model_cls.fit(
        target_fn=target_fn, dataset=ds_train,
        model_kwargs=model_kwargs,
        hyperparameter_search=True,
        n_iter_search=HPO_N_ITER_TREES,
        cv=GroupKFold(n_splits=HPO_CV_FOLDS),
        cv_group_by=KEY_YEAR,
        random_state=seed,
    )
    if 'best_params' in info:
        save_hp_cache(dataset_name, model_key, info['best_params'])
    return model


# ---------------------------------------------------------------------------
# Torch HPO (CNN1D / LSTM / Transformer): plain random search over a small
# space.
#   - Build a cutoff val set (last HPO_VAL_FRACTION of training years).
#   - For each trial: fit on (full ds_train minus cutoff val) using fit()'s
#     own internal random val for early stopping, then score MAE on the
#     external cutoff val. The two val sets serve different roles
#     (stopping vs selection) and don't overlap.
#   - Final fit (after HPO) uses the best config on the *full* ds_train.
# ---------------------------------------------------------------------------

def _split_train_for_hpo(ds_train, val_fraction: float):
    years_sorted = sorted(set(ds_train.years))
    cutoff_ix = int(len(years_sorted) * (1.0 - val_fraction))
    cutoff_ix = max(1, min(cutoff_ix, len(years_sorted) - 1))
    cutoff = years_sorted[cutoff_ix]
    years_trn = [y for y in years_sorted if y < cutoff]
    years_val = [y for y in years_sorted if y >= cutoff]
    return ds_train.select_years(years_trn), ds_train.select_years(years_val)


def _lstm_search_space(rng: np.random.Generator) -> Dict[str, Any]:
    return {
        'hidden_size':   int(rng.choice([32, 64, 128, 256])),
        'num_layers':    int(rng.choice([1, 2, 3])),
        'learning_rate': float(10 ** rng.uniform(-4, -2)),   # log-uniform [1e-4, 1e-2]
    }


def _transformer_search_space(rng: np.random.Generator) -> Dict[str, Any]:
    hidden_size = int(rng.choice([64, 128, 256]))
    # nhead must divide hidden_size
    valid_nheads = [n for n in [2, 4, 8] if hidden_size % n == 0]
    return {
        'hidden_size':    hidden_size,
        'num_layers':     int(rng.choice([1, 2, 3])),
        'nhead':          int(rng.choice(valid_nheads)),
        'dim_feedforward': 2 * hidden_size,
        'learning_rate':  float(10 ** rng.uniform(-4, -2)),
    }


def _cnn_search_space(rng: np.random.Generator) -> Dict[str, Any]:
    # CNN1D now uses exponential dilations (dilation_base ** layer_index). The
    # receptive field is  1 + (k-1) * (d^L - 1) / (d - 1). We constrain the
    # search to (kernel_size, num_layers) combinations whose RF >= 365 days,
    # since anything smaller can't see a full season.
    candidates = []
    for k in (3, 5):
        for L in range(4, 11):
            rf = 1 + (k - 1) * (2 ** L - 1)
            if rf >= 365:
                candidates.append((k, L))
    k, L = candidates[int(rng.integers(len(candidates)))]
    return {
        'hidden_size':   int(rng.choice([32, 64, 128])),
        'num_layers':    L,
        'kernel_size':   k,
        'dilation_base': 2,
        'learning_rate': float(10 ** rng.uniform(-4, -2)),
    }


# Per-model dispatcher: arch keys vs training keys. The HP cache stores a flat
# dict; this maps it back into the right argument bucket at fit time.
def _torch_dispatch(config: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    arch = {k: v for k, v in config.items() if k != 'learning_rate'}
    opt_kwargs = (
        dict(lr=float(config['learning_rate']), weight_decay=1e-4)
        if 'learning_rate' in config else None
    )
    return arch, opt_kwargs


def _torch_hpo(model_cls, target_fn, ds_train, search_space_fn, n_trials, seed):
    ds_trn_prime, ds_val_cutoff = _split_train_for_hpo(ds_train, HPO_VAL_FRACTION)
    if len(ds_val_cutoff) == 0 or len(ds_trn_prime) == 0:
        return None, []

    rng = np.random.default_rng(seed)
    trials = []

    for i in range(n_trials):
        config = search_space_fn(rng)
        arch, opt_kwargs = _torch_dispatch(config)
        t0 = time.time()
        try:
            model, _info = model_cls.fit(
                target_fn=target_fn, dataset=ds_trn_prime,
                model_kwargs=dict(data_keys=FEATURE_KEYS, **arch),
                seed=seed,
                optimizer_kwargs=opt_kwargs,
                verbose=False,
                **TORCH_TRAIN_KWARGS,
            )
            res = SingleTargetRegression.run(
                model=model,
                dataset_train=ds_trn_prime,
                dataset_test=ds_val_cutoff,
                target_fn=target_fn,
                run_name=f'_hpo_trial_tmp',
            )
            val_mae = float(res.compute_metrics()['test'].get('mae', float('inf')))
            err = None
        except Exception as exc:
            val_mae = float('inf')
            err = f'{type(exc).__name__}: {exc}'
        trials.append(dict(
            trial=i, config=config, val_mae=val_mae,
            seconds=round(time.time() - t0, 1), error=err,
        ))
        marker = '   '
        if err is None and val_mae == min(t['val_mae'] for t in trials if t['error'] is None):
            marker = '★  '
        print(f'      trial {i:2d}{marker}MAE_val={val_mae:.3f}  {config}  ({trials[-1]["seconds"]}s)')

    finite = [t for t in trials if t['error'] is None and np.isfinite(t['val_mae'])]
    if not finite:
        return None, trials
    best = min(finite, key=lambda t: t['val_mae'])
    return best['config'], trials


def _maybe_run_torch_hpo(model_cls, target_fn, ds_train, seed, dataset_name, model_key,
                         search_space_fn, default_arch: Dict[str, Any]):
    cached = None if _should_retune(model_key) else load_hp_cache(dataset_name, model_key)

    if cached is None and RUN_HPO_TORCH:
        print(f'    [{model_key}] running {HPO_N_TRIALS_TORCH} random-search trials...')
        best_config, trials = _torch_hpo(
            model_cls, target_fn, ds_train, search_space_fn,
            n_trials=HPO_N_TRIALS_TORCH, seed=seed,
        )
        save_hp_trials(dataset_name, model_key, trials)
        if best_config is not None:
            save_hp_cache(dataset_name, model_key, best_config)
            cached = best_config

    arch_kwargs = dict(default_arch)
    if cached is not None:
        arch, opt_kwargs = _torch_dispatch(cached)
        arch_kwargs.update(arch)
    else:
        opt_kwargs = None

    # Final fit on full ds_train with best (or default) config.
    model, _ = model_cls.fit(
        target_fn=target_fn, dataset=ds_train,
        model_kwargs=dict(data_keys=FEATURE_KEYS, **arch_kwargs),
        seed=seed,
        optimizer_kwargs=opt_kwargs,
        **TORCH_TRAIN_KWARGS,
    )
    return model


# ---------------------------------------------------------------------------
# Fit callables. Signature: (target_fn, ds_train, seed, ds_name, model_key) -> model
# ---------------------------------------------------------------------------

def fit_mean(target_fn, ds_train, seed, ds_name=None, model_key=None):
    model, _ = MeanModel.fit(target_fn=target_fn, dataset=ds_train)
    return model


def fit_linear(target_fn, ds_train, seed, ds_name=None, model_key=None):
    model, _ = LinearTrendModel.fit(target_fn=target_fn, dataset=ds_train)
    return model


def fit_rf(target_fn, ds_train, seed, ds_name='', model_key='RandomForest'):
    return _maybe_run_tree_hpo(RandomForestModel, target_fn, ds_train, seed, ds_name, model_key)


def fit_xgb(target_fn, ds_train, seed, ds_name='', model_key='XGBoost'):
    return _maybe_run_tree_hpo(XGBoostModel, target_fn, ds_train, seed, ds_name, model_key)


def fit_cnn(target_fn, ds_train, seed, ds_name='', model_key='CNN1D'):
    return _maybe_run_torch_hpo(
        CNN1DModel, target_fn, ds_train, seed, ds_name, model_key,
        search_space_fn=_cnn_search_space,
        default_arch=dict(),  # let the model use its own (num_layers=8, kernel=3, dilation_base=2)
    )


def fit_lstm(target_fn, ds_train, seed, ds_name='', model_key='LSTM'):
    return _maybe_run_torch_hpo(
        LSTMModel, target_fn, ds_train, seed, ds_name, model_key,
        search_space_fn=_lstm_search_space,
        default_arch=dict(hidden_size=64, num_layers=2),
    )


def fit_transformer(target_fn, ds_train, seed, ds_name='', model_key='Transformer'):
    return _maybe_run_torch_hpo(
        TransformerModel, target_fn, ds_train, seed, ds_name, model_key,
        search_space_fn=_transformer_search_space,
        default_arch=dict(hidden_size=64, num_layers=2, nhead=4, dim_feedforward=128),
    )


# Each entry: (model_class, fit_callable). The class is needed for load(),
# the callable for fit() — they would otherwise have different signatures.
MODELS: Dict[str, Tuple[Any, Callable[..., Any]]] = {
    'Mean':         (MeanModel,         fit_mean),
    'Linear':       (LinearTrendModel,  fit_linear),
    'RandomForest': (RandomForestModel, fit_rf),
    'XGBoost':      (XGBoostModel,      fit_xgb),
    'CNN1D':        (CNN1DModel,        fit_cnn),
    'LSTM':         (LSTMModel,         fit_lstm),
    'Transformer':  (TransformerModel,  fit_transformer),
}

print('HP cache dir:', HP_CACHE_DIR)
list(MODELS)

## 3. Helper: load + temporal 75/25 split

In [ ]:
def load_dataset(dataset_name: str) -> Dataset:
    calendar = Calendar(default_start=SEASON_START, default_length=SEASON_LENGTH)
    providers = [OpenMeteoFeatures(calendar=calendar, data_keys=FEATURE_KEYS)]
    ds = Dataset.load(dataset_name, calendar=calendar, feature_providers=providers)
    ds.download_features(verbose=False)
    return ds


def split_by_year_cutoff(ds: Dataset, train_size: float = SPLIT_SIZE) -> Tuple[Dataset, Dataset]:
    """Match `fit_eval`'s `--split_years cutoff --split_years_size 0.75`."""
    years_sorted = sorted(set(ds.years))
    ix = int(len(years_sorted) * train_size)
    cutoff = years_sorted[ix] if ix < len(years_sorted) else years_sorted[-1] + 1
    years_trn = [y for y in years_sorted if y < cutoff]
    years_tst = [y for y in years_sorted if y >= cutoff]
    return ds.select_years(years_trn), ds.select_years(years_tst)

## 4. Load the two datasets and verify the split

First run will download missing climate data for the cherry locations — this can take a minute or two but is cached for subsequent runs.

In [ ]:
datasets: Dict[str, Tuple[Dataset, Dataset, str]] = {}
for ds_name, target in DATASETS:
    print(f'\n--- {ds_name} ---')
    ds_full = load_dataset(ds_name)
    ds_trn, ds_tst = split_by_year_cutoff(ds_full)
    print(f'  total:   {len(ds_full):5d} samples,  {len(set(ds_full.years)):3d} years '
          f'({min(ds_full.years)}-{max(ds_full.years)})')
    print(f'  train:   {len(ds_trn):5d} samples,  {len(set(ds_trn.years)):3d} years')
    print(f'  test:    {len(ds_tst):5d} samples,  {len(set(ds_tst.years)):3d} years')
    datasets[ds_name] = (ds_trn, ds_tst, target)

## 5. Fit every model on every dataset and collect test MAE

In [ ]:
from pysephone.models.base import ModelException


def _should_force_retrain(model_key: str) -> bool:
    return FORCE_RETRAIN_ALL or model_key in FORCE_RETRAIN_MODELS


def _try_load(model_cls, run_name: str):
    # Return a loaded model or None if no cache exists / cache is broken.
    try:
        model, _ = model_cls.load(run_name)
        return model
    except (ModelException, FileNotFoundError, EOFError, ModuleNotFoundError):
        return None
    except Exception as exc:
        print(f'    cache load failed for {run_name}: {type(exc).__name__}: {exc} -- refitting')
        return None


rows = []
for ds_name, (ds_trn, ds_tst, target) in datasets.items():
    target_fn = lambda s, k=target: s[KEY_OBSERVATIONS][k]

    for model_key, (model_cls, fit_fn) in MODELS.items():
        run_name = f'jp_draft_{ds_name}_{model_key}_seed{SEED}'
        t0 = time.time()
        try:
            model = None
            if not _should_force_retrain(model_key):
                model = _try_load(model_cls, run_name)

            if model is None:
                source = 'fit'
                model = fit_fn(target_fn, ds_trn, SEED, ds_name, model_key)
                model.save(run_name)
            else:
                source = 'cache'

            result = SingleTargetRegression.run(
                model=model,
                dataset_train=ds_trn,
                dataset_test=ds_tst,
                target_fn=target_fn,
                run_name=run_name,
                extra_metadata=dict(dataset=ds_name, model=model_key, seed=SEED, source=source),
            )
            # Persist predictions so the downstream scatter cell can load them
            # back by run_name. Cheap (one CSV per split + a tiny JSON).
            result.save()
            metrics = result.compute_metrics()
            rows.append(dict(
                dataset=ds_name,
                model=model_key,
                source=source,
                mae_train=metrics['train'].get('mae', float('nan')),
                mae_test =metrics['test'].get('mae',  float('nan')),
                rmse_test=metrics['test'].get('rmse', float('nan')),
                r2_test  =metrics['test'].get('r2',   float('nan')),
                n_train  =metrics['train'].get('n', 0),
                n_test   =metrics['test'].get('n', 0),
                seconds  =round(time.time() - t0, 1),
                error    =None,
            ))
        except Exception as e:
            rows.append(dict(
                dataset=ds_name, model=model_key, source='-',
                mae_train=float('nan'), mae_test=float('nan'),
                rmse_test=float('nan'), r2_test=float('nan'),
                n_train=0, n_test=0,
                seconds=round(time.time() - t0, 1),
                error=f'{type(e).__name__}: {e}',
            ))
        last = rows[-1]
        if last['error'] is None:
            status = f"[{last['source']:5s}] MAE_test={last['mae_test']:.2f}"
        else:
            status = f"FAILED ({last['error']})"
        print(f"  [{ds_name:24s}] {model_key:12s}  {status}  ({last['seconds']}s)")

results = pd.DataFrame(rows)
results

## 6. Compact MAE table (test set)

In [ ]:
table = (
    results.pivot(index='model', columns='dataset', values='mae_test')
           .reindex(index=list(MODELS))
           .reindex(columns=[name for name, _ in DATASETS])
           .round(2)
)
table

### 6a. Cached hyperparameters (tree models)

Shows the contents of `outputs/hyperparams/<dataset>_<model>.json` — i.e. the `best_params` the most recent search settled on. Empty rows mean the cache file does not exist for that pair (either HPO was turned off, or this dataset/model combination was never tuned).

In [ ]:
TUNED_MODELS = ['RandomForest', 'XGBoost', 'CNN1D', 'LSTM', 'Transformer']
hp_rows = []
for ds_name, _ in DATASETS:
    for model_key in TUNED_MODELS:
        cached = load_hp_cache(ds_name, model_key)
        if cached is None:
            hp_rows.append(dict(dataset=ds_name, model=model_key, params='(no cache)'))
        else:
            hp_rows.append(dict(
                dataset=ds_name, model=model_key,
                params=', '.join(f'{k}={v}' for k, v in sorted(cached.items())),
            ))
pd.DataFrame(hp_rows)

### 6b. Torch-model random-search trial logs

Per (dataset, model), shows the val-MAE achieved by each of the
`HPO_N_TRIALS_TORCH` random-search trials. A horizontal red line marks the
selected (best) trial. Rerunning this cell after `FORCE_RETUNE_MODELS={'LSTM'}`
followed by the run-loop cell will overwrite the LSTM trials JSON and refresh
the plot.

In [ ]:
TORCH_TUNED = ['CNN1D', 'LSTM', 'Transformer']
pairs = [(ds, m) for ds, _ in DATASETS for m in TORCH_TUNED
         if load_hp_trials(ds, m) is not None]

if not pairs:
    print('No torch trial logs yet — run section 5 first with RUN_HPO_TORCH=True.')
else:
    n = len(pairs)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.2 * nrows), squeeze=False)
    for ax, (ds_name, model_key) in zip(axes.flat, pairs):
        trials = load_hp_trials(ds_name, model_key)
        ix = [t['trial'] for t in trials]
        val = [t['val_mae'] if np.isfinite(t['val_mae']) else np.nan for t in trials]
        ax.plot(ix, val, marker='o', linestyle='-', color='steelblue')
        finite = [v for v in val if np.isfinite(v)]
        if finite:
            best = min(finite)
            ax.axhline(best, color='crimson', linestyle='--', linewidth=1,
                       label=f'best = {best:.2f}')
            ax.legend(loc='upper right', fontsize=9)
        ax.set_title(f'{ds_name} — {model_key}')
        ax.set_xlabel('Trial')
        ax.set_ylabel('Val MAE')
    for ax in axes.flat[len(pairs):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

## 7. Bar plot — test MAE per model, side-by-side

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
models_order = list(MODELS)
x = np.arange(len(models_order))
width = 0.38

for i, (ds_name, _) in enumerate(DATASETS):
    vals = [
        results.query('dataset == @ds_name and model == @m')['mae_test'].iloc[0]
        if not results.query('dataset == @ds_name and model == @m').empty else float('nan')
        for m in models_order
    ]
    ax.bar(x + (i - 0.5) * width, vals, width=width, label=ds_name)
    for xi, v in zip(x + (i - 0.5) * width, vals):
        if np.isfinite(v):
            ax.text(xi, v, f'{v:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models_order, rotation=20, ha='right')
ax.set_ylabel('Test MAE (days)')
ax.set_title('Japan cherry — Yedoensis vs Sargentii (draft)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Best-model scatter (predicted vs observed)

For each dataset, picks the model with the lowest test MAE and plots its predicted vs observed bloom DOY on the held-out test set.

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(5 * len(DATASETS), 5), squeeze=False)
for ax, (ds_name, _) in zip(axes[0], DATASETS):
    sub = results.query('dataset == @ds_name and error.isna()', engine='python')
    if sub.empty:
        ax.set_title(f'{ds_name}\n(no successful runs)'); continue
    best = sub.sort_values('mae_test').iloc[0]
    run = SingleTargetRegression.load(f'jp_draft_{ds_name}_{best["model"]}_seed{SEED}')
    if run.df_test.empty:
        ax.set_title(f'{ds_name}\n(no test predictions)'); continue
    obs  = run.df_test['observed_doy'].values
    pred = run.df_test['predicted_doy'].values
    lo = min(obs.min(), pred.min()); hi = max(obs.max(), pred.max())
    pad = (hi - lo) * 0.05
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], '--', color='grey', lw=1)
    ax.scatter(obs, pred, s=12, alpha=0.6)
    ax.set_xlabel('Observed DOY')
    ax.set_ylabel('Predicted DOY')
    ax.set_title(f'{ds_name} — best: {best["model"]}  (MAE_test = {best["mae_test"]:.2f})')
plt.tight_layout()
plt.show()

## 9. Statistical comparison (Friedman + Nemenyi)

Uses the existing [`pysephone.evaluation.model_comparison`](../src/pysephone/evaluation/model_comparison.py) procedure, which:

1. Aggregates the cached per-run `SingleTargetRegression` results into a
   `(datasets × models)` MAE scores matrix
2. Runs the **Friedman omnibus** to test whether any model differs from the rest
3. Runs the **Nemenyi post-hoc** for pairwise comparisons (built-in studentized-range correction — no separate multiple-testing decision required)
4. Renders three plots: scores heatmap, Nemenyi p-value heatmap, per-dataset rank distribution

**Caveat for this draft**: with only **2 datasets** and **1 seed**, the Friedman test is severely underpowered. Friedman/Nemenyi were designed for "many datasets, few models" benchmarks (Demšar 2006 recommends N ≥ 5 datasets). Interpret the omnibus p-value with caution; the ranking and Nemenyi matrix are still informative as a descriptive summary.

In [ ]:
from pysephone.evaluation.model_comparison import (
    EvaluationRun,
    compare_models,
    plot_score_heatmap,
    plot_nemenyi_heatmap,
    plot_rank_distribution,
)

# All models from the MODELS registry participate in the statistical comparison.
COMPARISON_MODELS = list(MODELS)

# Build EvaluationRuns from the on-disk cache produced by Section 5. Skips any
# (model, dataset) cell that didn't finish (e.g. failed runs in `results`).
comparison_runs = []
for ds_name, _ in DATASETS:
    for model_key in COMPARISON_MODELS:
        run_name = f'jp_draft_{ds_name}_{model_key}_seed{SEED}'
        try:
            eval_result = SingleTargetRegression.load(run_name)
        except Exception as exc:
            print(f'  skipping {run_name}: {type(exc).__name__}: {exc}')
            continue
        comparison_runs.append(EvaluationRun(
            eval_result=eval_result,
            model_key=model_key,
            dataset_key=ds_name,
            seed=SEED,
        ))

print(f'\nLoaded {len(comparison_runs)} runs across '
      f'{len(DATASETS)} datasets x {len(COMPARISON_MODELS)} models.')

report = compare_models(
    comparison_runs,
    model_keys=COMPARISON_MODELS,
    dataset_keys=[name for name, _ in DATASETS],
    seeds=[SEED],
    metric='mae',
    split='test',
    alpha=0.05,
    order='ascending',  # lower MAE is better
)

print()
print(report.summary())

In [ ]:
from pysephone.evaluation.model_comparison import (
    autorank_report,
    plot_critical_difference,
)

# Plots from the existing evaluation.model_comparison module
fig_scores  = plot_score_heatmap(report.scores, metric=report.metric)
fig_nemenyi = plot_nemenyi_heatmap(report)
fig_ranks   = plot_rank_distribution(report.scores, order=report.order)

# Critical-difference diagram (Demsar 2006). Built on top of `autorank`. With
# only 2 datasets the CD diagram is barely meaningful — autorank may refuse to
# produce it, in which case we surface the message rather than swallow it.
try:
    ar = autorank_report(report.scores, alpha=report.alpha, order=report.order)
    fig_cd = plot_critical_difference(ar)
except Exception as exc:
    print(f'Critical-difference diagram skipped: {type(exc).__name__}: {exc}')

plt.show()

## Notes

* This is a **single-seed draft**. The manuscript reports MAE±SD over 5 seeds — to extend this, wrap the run loop in a seed loop and aggregate.
* `RandomForestModel` and `XGBoostModel` here use **fixed defaults** (no hyperparameter search). Set `hyperparameter_search=True` to enable `RandomizedSearchCV`.
* Only `temperature_2m_mean` is used as a feature; the manuscript also includes day length. Add `'daylight_duration'` to `FEATURE_KEYS` to match the manuscript spec.
* Datasets `GMU_Cherry_Japan_Y` (yedoensis) and `GMU_Cherry_Japan_S` (sargentii, **added in this draft** to `src/pysephone/dataset/registry/gmu_cherry.py`) both target the same observation code `gmu_0`.